In [6]:
import argparse, os, random, numpy as np, time, json
from pathlib import Path
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from collections import Counter

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_logits, all_y = [], []
    for x,y in dataloader:
        x = x.to(device)
        logits = model(x)
        all_logits.append(logits.cpu()); all_y.append(y)
    logits = torch.cat(all_logits)
    y_true = torch.cat(all_y).numpy()
    y_pred = logits.argmax(1).numpy()
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    return acc, f1



In [ ]:
def make_loaders(data_dir, img_size=224, batch_size=64, seed=42, normalize=False, min_class_count=12):
    
    # t = [transforms.Resize((img_size, img_size)), transforms.ToTensor()]
    # if normalize:
    #     t += [transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])]
    # tfm = transforms.Compose(t)

    # Add Randomness (Augmentation)
    train_t = [
        transforms.Resize((256, 256)),        # Resize slightly larger
        transforms.RandomCrop((img_size, img_size)), # Random crop
        transforms.RandomHorizontalFlip(),    # Flip left/right
        transforms.RandomRotation(15),        # Rotate slightly
        transforms.ColorJitter(brightness=0.1, contrast=0.1), # Light changes
        transforms.ToTensor()
    ]
    
    val_t = [
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ]
    
    if normalize:
        stats = {'mean':[0.485,0.456,0.406], 'std':[0.229,0.224,0.225]}
        train_t.append(transforms.Normalize(**stats))
        val_t.append(transforms.Normalize(**stats))
        
    train_tfm = transforms.Compose(train_t)
    val_tfm = transforms.Compose(val_t)

    ds_full = datasets.ImageFolder(data_dir, transform=val_tfm)
    
    targets_full = np.array(ds_full.targets)
    class_counts = Counter(targets_full)
    
    valid_class_indices = [cls for cls, count in class_counts.items() if count >= min_class_count]
    
    filtered_indices_list = [i for i, target in enumerate(targets_full) if target in valid_class_indices]
    filtered_indices = np.array(filtered_indices_list) 
    
    y = targets_full[filtered_indices]

    print(f"Original size: {len(targets_full)}. Filtered size (min_count={min_class_count}): {len(y)}")
    
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_rel, tmp_idx_rel = next(sss1.split(np.zeros(len(y)), y))
    
    train_idx = filtered_indices[train_idx_rel]
    tmp_idx = filtered_indices[tmp_idx_rel]
    
    y_tmp = targets_full[tmp_idx]
    
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_rel_idx, test_rel_idx = next(sss2.split(np.zeros(len(y_tmp)), y_tmp))
    
    val_idx = tmp_idx[val_rel_idx]
    test_idx = tmp_idx[test_rel_idx]
    
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    all_paths = [os.path.abspath(p) for p, _ in ds_full.samples]
    
    split = {
        "paths": all_paths,                    
        "labels": ds_full.targets,                    
        "train_idx": train_idx.tolist(),
        "val_idx": val_idx.tolist(),
        "test_idx": test_idx.tolist(),
        "img_size": img_size,
        "normalized": bool(normalize),
    }
    with open(out_dir / "split.json", "w") as f:
        json.dump(split, f)
    print(f"Dataset split lists saved to {out_dir/'split.json'}")

    train_subset = Subset(ds_full, train_idx)
    train_subset.dataset.transform = train_tfm # Force train transform
    
    dl_train = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    dl_val   = DataLoader(Subset(ds_full, val_idx),   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    dl_test  = DataLoader(Subset(ds_full, test_idx),  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    return ds_full, dl_train, dl_val, dl_test

In [8]:
def build_model(num_classes, load_ckpt_path=None):
    if load_ckpt_path:
        print(f"Loading baseline model from: {load_ckpt_path}")
        # Load the checkpoint
        ckpt = torch.load(load_ckpt_path, map_location="cpu")
        
        # --- Infer number of classes from the checkpoint ---
        ckpt_num_classes = 0
        w = ckpt["model"].get("classifier.3.weight", None)
        if w is None:
            w = ckpt["model"].get("classifier.1.weight", None) # Fallback
        
        if w is not None:
            ckpt_num_classes = int(w.shape[0])
        else:
            # Fallback for old/unknown checkpoints
            classes = ckpt.get("classes", [])
            if classes and isinstance(classes, list):
                ckpt_num_classes = len(classes)
            else:
                raise ValueError("Could not determine number of classes from checkpoint.")
                
        print(f"Checkpoint model has {ckpt_num_classes} classes.")

        # --- Build model structure and load weights ---
        m = models.mobilenet_v3_small(weights=None)
        in_features = m.classifier[3].in_features
        m.classifier[3] = nn.Linear(in_features, ckpt_num_classes)
        
        # Load the saved weights
        m.load_state_dict(ckpt["model"])
        print("Successfully loaded baseline model weights.")

        # --- Replace the head ---
        print(f"Replacing head for new dataset with {num_classes} classes.")
        m.classifier[3] = nn.Linear(in_features, num_classes)
        
    else:
        print("Building new model from ImageNet (MobileNet_V3_Small_Weights.IMAGENET1K_V1)")
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        in_features = m.classifier[3].in_features
        m.classifier[3] = nn.Linear(in_features, num_classes)
        
    return m

In [9]:
def run_fine_tuning(data_dir, load_ckpt_path, out_dir, img_size, batch_size, epochs, lr, normalize, seed):
    
    seed_all(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Data loaders are created from the PlantDoc directory
    ds, dl_train, dl_val, dl_test = make_loaders(data_dir, img_size, batch_size, seed, normalize)
    num_classes = len(ds.classes)
    
    # Model is built, loading the PV checkpoint and replacing the head
    model = build_model(num_classes, load_ckpt_path).to(device)

    params = [
        # Group 1: The Backbone (Pre-trained features) -> Keep Low LR (e.g. 1e-5)
        {'params': model.features.parameters(), 'lr': lr}, 
        
        # Group 2: The Classifier (New Random Head) -> Use High LR (e.g. 1e-3)
        {'params': model.classifier.parameters(), 'lr': 1e-3} 
    ]
    opt = optim.Adam(params, betas=(0.5, 0.99))
    
    # Optimizer and Loss
    # Adam, betas=(0.5, 0.99) were used in the paper
    # opt = optim.Adam(model.parameters(), lr=lr, betas=(0.5, 0.99))
    ce = nn.CrossEntropyLoss()

    best_val = -1; best_path = Path(out_dir)/'mobilenetv3small_finetune_best.pt'
    best_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x, y in tqdm(dl_train, desc=f"Epoch {epoch}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = ce(logits, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)

        # Validation
        val_acc, val_f1 = evaluate(model, dl_val, device)
        if val_acc > best_val:
            best_val = val_acc
            torch.save({'model': model.state_dict(), 'classes': ds.classes}, best_path)

        if epoch % 10 == 0 or epoch == 1:
            print(f"[{epoch}] train_loss={(total_loss/len(dl_train.dataset)):.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f}")

    # Load best and evaluate on test set
    ckpt = torch.load(best_path, map_location='cpu')
    model.load_state_dict(ckpt['model']); model.to(device)

    test_acc, test_f1 = evaluate(model, dl_test, device)
    print(f"--- FINE-TUNE TEST RESULTS ---")
    print(f"TEST acc={test_acc:.4f} | TEST macro-F1={test_f1:.4f}")
    final_path = Path(out_dir)/'mobilenetv3small_final.pt'
    torch.save({'model': model.state_dict(), 'classes': ds.classes}, final_path)
    print(f"Saved best to: {best_path}\nSaved final to: {final_path}")
    return model

In [ ]:
DATA_DIR = "PlantDoc-Dataset/train/"
LOAD_CKPT_PATH = "runs/frontiers2023/run2/mobilenetv3small_best.pt"
OUT_DIR = "runs/finetune-pd/run1"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-5
SEED = 42
NORMALIZE = True

In [13]:
model = run_fine_tuning(
    data_dir=DATA_DIR,
    load_ckpt_path=LOAD_CKPT_PATH,
    out_dir=OUT_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    normalize=NORMALIZE,
    seed=SEED,
)

Original size: 2336. Filtered size (min_count=12): 2334
Dataset split lists saved to runs/finetune-pd/run2/split.json
Loading baseline model from: runs/frontiers2023/run2/mobilenetv3small_best.pt
Checkpoint model has 38 classes.
Successfully loaded baseline model weights.
Replacing head for new dataset with 28 classes.


[1] train_loss=3.0468 | val_acc=0.2833 | val_f1=0.1853


[10] train_loss=1.0082 | val_acc=0.4378 | val_f1=0.4086


[20] train_loss=0.5138 | val_acc=0.5064 | val_f1=0.4821
--- FINE-TUNE TEST RESULTS ---
TEST acc=0.5085 | TEST macro-F1=0.4657
Saved best to: runs/finetune-pd/run2/mobilenetv3small_finetune_best.pt
Saved final to: runs/finetune-pd/run2/mobilenetv3small_final.pt
